# GuideLLM Canopy 백엔드 벤치마크 🚀

이 노트북에서는 Canopy 엔드포인트를 지원하도록 포크한 GuideLLM으로 **Canopy 백엔드**를 테스트합니다(엄밀히 말하면 Canopy처럼 텍스트를 입력받아 스트리밍으로 응답하는 모든 엔드포인트를 지원하지만, 확실히 하기 위해 Canopy인지 확인하는 검사를 추가했습니다).


수정된 버전의 GuideLLM을 사용합니다. 관심이 있다면 여기에서 확인할 수 있습니다: [https://github.com/RHRolun/guidellm](https://github.com/RHRolun/guidellm)

## 벤치마크 설정 🎯

모델 이름 앞에 "canopy_"를 붙이면 Canopy 백엔드를 지원하도록 자동으로 활성화됩니다.

In [ ]:
# ---- 대상 및 엔드포인트 설정 ----
base_url = "http://canopy-backend:8000"
endpoint = "/summarization"

# 엔드포인트 경로를 포함한 전체 대상 URL
target = f"{base_url}{endpoint}"  # 예: "https://...com/summarization"
model = "canopy_system"  # "canopy_"로 시작하는 모델은 Canopy 지원을 자동으로 활성화합니다
processor = "RedHatAI/Llama-3.2-3B-Instruct-quantized.w8a8"  # 토큰화에는 실제 HuggingFace 모델을 사용합니다. 여기서는 Canopy의 기반 모델을 사용합니다
output_path = "canopy-benchmark-summarize-endpoint.yaml"

print(f"Base URL: {base_url}")
print(f"Endpoint: {endpoint}")
print(f"Full target: {target}")
print(f"Processor: {processor}")
print(f"Output file: {output_path}")

## 에뮬레이션 워크로드 📦

기준 처리량(throughput) 테스트를 위해 토큰 크기가 고정된 프롬프트와 출력을 에뮬레이션합니다.

In [ ]:
# ---- 워크로드 설정 ----
data = {
    "type": "emulated",
    "prompt_tokens": 512,
    "output_tokens": 128,
}
rate_type = "synchronous"   # 요청 속도 유형
max_seconds = 60            # 전체 실행 시간(초)

backend_args = {}  # 별도의 backend args가 필요 없습니다. Canopy 지원이 내장되어 있습니다
extra_flags = ""  # 추가 플래그가 있으면 여기에 넣습니다

## 명령어 만들기 👀

여기서는 Canopy 백엔드용 GuideLLM 명령어를 생성하고, 무엇을 실행하는지 알 수 있도록 출력합니다.  
이 명령어는 나중에 평가를 프로덕션화할 때 사용합니다.

In [ ]:
import json, shlex

def build_canopy_guidellm_command(
    target: str,
    model: str,  # canopy_summarize 또는 canopy_info_search
    processor: str,
    data: dict,
    output_path: str,
    rate_type: str,
    max_seconds: int,
    backend_args: dict = None,
    extra_flags: str = "",
):
    data_json = json.dumps(data, separators=(",", ":"))
    
    cmd_list = [
        "guidellm", "benchmark",
        f"--target={target}",
        f"--model={model}",  # canopy 모델 이름 사용("canopy_"로 시작)
        f"--processor={processor}",
        f"--backend-type=openai_http",  # Canopy를 기본 지원하는 OpenAI 백엔드
        f"--data={data_json}",
        f"--output-path={output_path}",
        f"--rate-type={rate_type}",
        f"--max-seconds={int(max_seconds)}",
    ]
    
    # backend-args가 있을 때만 추가
    if backend_args:
        backend_args_json = json.dumps(backend_args, separators=(",", ":"))
        cmd_list.append(f"--backend-args={backend_args_json}")
    
    if extra_flags.strip():
        cmd_list.extend(extra_flags.strip().split())

    # 보기 좋은 셸 문자열
    parts = [
        "guidellm benchmark",
        f"--target={shlex.quote(target)}",
        f"--model={model}",
        f"--processor={shlex.quote(processor)}",
        f"--backend-type=openai_http",
        f"--data='{data_json}'",
        f"--output-path={shlex.quote(output_path)}",
        f"--rate-type={shlex.quote(rate_type)}",
        f"--max-seconds={int(max_seconds)}",
    ]
    
    # backend-args가 있을 때만 표시용 문자열에 추가
    if backend_args:
        backend_args_json = json.dumps(backend_args, separators=(",", ":"))
        parts.insert(-4, f"--backend-args='{backend_args_json}'")  # data 앞에 삽입
    
    if extra_flags.strip():
        parts.append(extra_flags)

    cmd_shell = " ".join(parts)
    return cmd_list, cmd_shell

cmd_list, cmd_shell = build_canopy_guidellm_command(
    target, model, processor, data, output_path, rate_type, max_seconds, backend_args, extra_flags
)

print(f"Canopy {endpoint} endpoint benchmark command (with native support):\n")
print(cmd_shell)
print("\nSubprocess list:\n")
print(cmd_list)

## 벤치마크 실행하기 ▶️

Canopy 백엔드를 대상으로 벤치마크를 실행하고 로그를 스트리밍합니다. 약 2분 정도 걸릴 수 있습니다 #잠시만기다려주세요

In [ ]:
import shutil, subprocess, sys, os

print(f"🚀 Starting canopy {endpoint} endpoint benchmark with native support...\n")
print(f"Target: {target}")
print(f"Endpoint: {endpoint}")
print(f"Model: {model} (enables native Canopy support)")
print(f"Duration: {max_seconds}s\n")

if shutil.which("guidellm") is None:
    print("⚠️  guidellm command not found in PATH, trying python module...")
    cmd_prefix = ["python", "-m", "guidellm"]
else:
    cmd_prefix = ["guidellm"]

# guidellm을 직접 실행합니다. 기본 지원 덕분에 래퍼가 필요 없습니다
process = subprocess.Popen([
    *cmd_prefix, "benchmark"
] + cmd_list[2:], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

for line in process.stdout:
    sys.stdout.write(line)
ret = process.wait()

print(f"\n✅ Benchmark completed. Exit code: {ret}")
if ret != 0:
    print("❌ Non-zero exit code. Review the logs above.")
    print("   Common issues:")
    print("   - Canopy service not accessible")
    print("   - Endpoint feature not enabled") 
    print("   - Network connectivity issues")
    print("   - Model name doesn't start with 'canopy_'")
else:
    print(f"📊 Results saved to: {output_path}")

## 결과 확인하기 📝

결과가 꽤 길지만, 다음은 그 요약입니다.

In [ ]:
from pathlib import Path

p = Path(output_path)
if p.exists():
    print(f"📋 Results: {p.resolve()}")
    print(f"📊 Benchmarked: Canopy {endpoint} endpoint (native support)")
    print(f"🎯 Target: {target}")
    print(f"⏱️  Duration: {max_seconds}s\n")
    
    try:
        print("--- First 200 lines ---\n")
        with p.open("r", encoding="utf-8", errors="replace") as f:
            for i, line in enumerate(f):
                if i >= 200:
                    print("... (truncated)")
                    break
                print(line.rstrip("\n"))
    except Exception as e:
        print(f"Preview error: {e}")
else:
    print(f"❌ Results not found: {p}")
    print("   Check the benchmark command and logs above.")

이제 결과를 해석하는 방법을 알아보겠습니다.

## GuideLLM 벤치마크 지표 이해하기 📊

보시다시피 GuideLLM은 훑어보기 어려울 정도로 **아주 많은** 출력을 생성합니다.  
다음은 살펴보면 좋은 핵심 성능 지표와 각 지표의 적정 범위입니다.  
아래 셀을 실행하면 우리가 얻은 결과를 볼 수 있으며, 모든 지표를 직접 살펴보고 싶다면 `canopy-benchmark-summarize-endpoint.yaml`에서 확인할 수 있습니다.

**🚀 Time to First Token (TTFT, 첫 토큰까지의 시간)**
- **측정 대상**: 사용자가 응답을 처음 보기까지 기다리는 시간
- **적정 범위**: < 500ms (매우 좋음 < 200ms)
- **영향**: 사용자가 체감하는 반응성

**⚡ Output Tokens per Second (초당 출력 토큰 수)** 
- **측정 대상**: 스트리밍 중 텍스트 생성 속도
- **적정 범위**: 20-100 tokens/sec (모델 크기에 따라 다름)
- **영향**: 사용자에게 텍스트가 나타나는 속도

**📈 Requests per Second (초당 요청 수)**
- **측정 대상**: 시스템이 초당 처리하는 완료된 요청 수
- **적정 범위**: 사용 사례에 따라 다름 (0.1-10+ req/sec)
- **영향**: 시스템 처리량과 수용 가능한 사용자 수

**🎯 Request Latency (요청 지연 시간)**
- **측정 대상**: 요청 시작부터 완료까지의 전체 시간
- **적정 범위**: 긴 응답의 경우 < 30초
- **영향**: 전반적인 사용자 경험

**⚙️ Inter-Token Latency (토큰 간 지연 시간)**
- **측정 대상**: 토큰 생성의 일관성(토큰 사이의 밀리초)
- **적정 범위**: 10-50ms (낮을수록 일관적)
- **영향**: 스트리밍 텍스트의 부드러움

**📊 Success Rate (성공률)**
- **측정 대상**: 성공적으로 완료된 요청의 비율
- **적정 범위**: > 95% (프로덕션 요구 사항)
- **영향**: 시스템 안정성

In [ ]:
from metrics_analyzer import analyze_benchmark_results
analyze_benchmark_results(output_path=output_path)

## HTML 리포트 🕸️

아래 명령어는 벤치마크 결과가 담긴 YAML 파일을 시각적인 HTML 리포트로 생성해줍니다. 명령어 실행 후, 워크벤치 화면 왼쪽을 보시면 'canopy-benchmark-summarize-endpoint.html' 파일이 새로 생성된 것을 확인하실 수 있습니다. 그 HTML 파일을 다운로드(마우스 오른쪽 클릭 > Download)한 뒤 브라우저에서 열어 대화형 차트와 지표를 살펴보세요.

In [ ]:
from benchmark_report import generate_html_report

generate_html_report(output_path, "canopy-benchmark-summarize-endpoint.html")